In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
import scanpy as sc
import squidpy as sq
import cv2
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import numpy as np

from pathlib import Path

# import sys
# sys.path.append('/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/workflow/scripts')
# import coda
# import readwrite
# cfg = readwrite.config()

import sys
sys.path.append("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1")
from norkin_organoid.code.get_embeddings import (
    NorkinOrganoidDataset,
    plot_organoids,
    get_morphological_features,
)
from norkin_organoid.code.extract_organoid import (
    extract_reoriented_optimized,
    get_microscopy, 
    get_transform_matrix,
    write_pyramidal_ome_tiff,
    save_png_preview,
    extract_lazyslide_features,
    pad_image_evenly
)

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# ["run_1"]="1HVQ 1CNN 077I 1GAA 1J25 131N OWJ3 14PT"
# ["run_2"]="169V 1BI7 1CI5 1FMS 12NM OLR9 1GVB 1GNS"
ALIGNED_PATIENT_IDS = ["1CNN", "1GAA", "1GVB", "1J25", "14PT", "131N"]

PATIENT_ID = "1CNN" # "1CNN" for example; "all" if all.
ORGANOID_CELL_MAPPING_PATH = "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/ORGANOID_IDs_Xenium_norkin.csv"
ALIGNMENTS_ROOT_PATH = lambda patient_id: f"/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/alignments/{patient_id}_qupath_alignment_files"
ORGANOID_COUNT_THRESHOLD = 20
SCALE_FACTOR = 1 / 0.2125
VISUALIZATION = False and PATIENT_ID != "all"

In [ ]:
patient_id = "1CNN"
organoid_id = "proseg_expected_CRC_PDO_hImmune_v1_dapi_1CNN_output-XETG00059__0003381__1CNN__20250505__170803_comp10"

tmp_pth = "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/organoids_h&e/tmp"

dataset = NorkinOrganoidDataset(standardize_scale=False, scale=False, fill=True)

: 

# Analysis 1: Morphological Metric Redundancy

This analysis concerns the morphological metrics that are redundant; in other words, which subset of morphological metrics can best predict the remaining?

In [ ]:
masks = np.array(list(dataset.organoid_masks.values()))
metrics = get_morphological_features(masks)

In [ ]:
metrics_df = pd.DataFrame(metrics)
metrics_df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# Alternative version with cleaner annotations (less cluttered)
def minimal_analysis_clean(df):
    features = df.columns.tolist()
    n = len(features)
    df_std = (df - df.mean()) / df.std()
    
    results = []
    best_combinations = []
    
    for k in range(1, n):
        trials = [np.random.choice(features, k, replace=False) for _ in range(150)]
        best_r2 = -1
        best_kept = None
        
        for kept in trials:
            kept_list = list(kept)
            predicted_features = [f for f in features if f not in kept_list]
            
            r2_scores = []
            for f in predicted_features:
                model = LinearRegression()
                model.fit(df_std[kept_list], df_std[f])
                y_pred = model.predict(df_std[kept_list])
                r2_scores.append(r2_score(df_std[f], y_pred))
            
            avg_r2 = np.mean(r2_scores)
            if avg_r2 > best_r2:
                best_r2 = avg_r2
                best_kept = kept_list
        
        lost_variance = 1 - best_r2
        results.append(lost_variance)
        best_combinations.append({
            'k': k,
            'kept': best_kept,
            'dropped': [f for f in features if f not in best_kept],
            'lost_variance': lost_variance
        })
    
    # Create plot with selective annotations (every few points to avoid clutter)
    plt.figure(figsize=(12, 8))
    k_values = range(1, n)
    plt.plot(k_values, results, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Number of Features Kept')
    plt.ylabel('Lost Variance (1 - R²)')
    plt.title('Lost Variance vs Features Kept\n(Key Dropped Features Annotated)')
    plt.grid(True, alpha=0.3)
    
    # Annotate only key points to avoid clutter
    annotation_points = [0, len(best_combinations)//4, len(best_combinations)//2, 
                        3*len(best_combinations)//4, -1]  # First, quarter, half, three-quarter, last
    
    for idx in annotation_points:
        if idx < len(best_combinations):
            combo = best_combinations[idx]
            k = combo['k']
            lost_var = combo['lost_variance']
            dropped = combo['dropped']
            
            # Shorten label if too many features
            if len(dropped) > 4:
                dropped_label = f"k={k}\nDropped: {', '.join(dropped[:3])}..."
            else:
                dropped_label = f"k={k}\nDropped: {', '.join(dropped)}"
            
            plt.annotate(dropped_label, 
                        xy=(k, lost_var), 
                        xytext=(10, 30),
                        textcoords='offset points',
                        ha='left', 
                        va='bottom',
                        fontsize=9,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.8),
                        arrowprops=dict(arrowstyle='->', color='gray'))
    
    plt.tight_layout()
    plt.show()
    
    # Print full details in console
    print("\n" + "="*50)
    print("DETAILED FEATURE DROPPING SEQUENCE")
    print("="*50)
    for combo in best_combinations:
        print(f"\nk={combo['k']}: Lost variance = {combo['lost_variance']:.4f}")
        print(f"Kept ({len(combo['kept'])}): {combo['kept']}")
        print(f"Dropped ({len(combo['dropped'])}): {combo['dropped']}")
    
    return results, best_combinations

# Run analysis
results, combinations = minimal_analysis_clean(metrics_df)

In [ ]:
data_matrix = metrics_df.values

# Analysis 2: Morphological Metric Clusters Reprojected into Xenium

This analysis concerns getting the masks clustered, with cluster IDs, that are extracted and relocalized with cluster IDs.

## UMAP from morphological metrics

# Analysis 3: Morphological Metric 1D Visualization

Finally, we are interested in visualizing the morphological metrics along a 1 dimensional axis. 

In [ ]:
masks.shape

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import minmax_scale

def create_quantile_image_line(metrics_df, masks, n_samples=10, figsize=(15, 8)):
    """
    Create 1D plots mapping linspace to quantiles for each metric with mask images
    
    Parameters:
    -----------
    metrics_df : pandas DataFrame
        DataFrame with metrics as columns
    masks : numpy array
        Array of shape (n_samples, 224, 224) containing mask images
    n_samples : int
        Number of sample points to display along the linspace
    figsize : tuple
        Figure size
    """
    n_metrics = len(metrics_df.columns)
    
    # Create subplots - one for each metric
    fig, axes = plt.subplots(n_metrics, 1, figsize=(figsize[0], figsize[1] * n_metrics / 3))
    if n_metrics == 1:
        axes = [axes]
    
    for idx, (metric, ax) in enumerate(zip(metrics_df.columns, axes)):
        # Get metric values and corresponding indices
        metric_values = metrics_df[metric].values
        indices = metrics_df.index.values
        
        # Create linspace from min to max of metric
        x_min, x_max = metric_values.min(), metric_values.max()
        x_linspace = np.linspace(x_min, x_max, n_samples)
        
        # Find closest actual data points for each linspace value
        sample_indices = []
        sample_values = []
        
        for x_val in x_linspace:
            # Find index of closest actual data point
            closest_idx = np.argmin(np.abs(metric_values - x_val))
            sample_indices.append(indices[closest_idx])
            sample_values.append(metric_values[closest_idx])
        
        # Set up the plot
        ax.set_xlim(x_min - 0.1*(x_max-x_min), x_max + 0.1*(x_max-x_min))
        ax.set_ylim(-0.5, 0.5)
        
        # Draw the number line
        ax.axhline(y=0, color='black', linewidth=2)
        
        # Plot points and add mask images
        for i, (x_val, sample_idx) in enumerate(zip(sample_values, sample_indices)):
            # Plot point on number line
            ax.plot(x_val, 0, 'ro', markersize=8, zorder=3)
            
            # Add value label
            ax.text(x_val, 0.15, f'{x_val:.2f}', 
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
            
            # Add mask image above the point
            if sample_idx < len(masks):
                img = masks[sample_idx]
                imagebox = OffsetImage(img, cmap='viridis', zoom=0.3)  # Adjust zoom as needed
                ab = AnnotationBbox(imagebox, (x_val, 0.3), 
                                  frameon=True, 
                                  boxcoords="data",
                                  pad=0.1,
                                  arrowprops=dict(arrowstyle="->", color='red'))
                ax.add_artist(ab)
        
        # Customize the subplot
        ax.set_title(f'Metric: {metric}', fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Metric Value', fontsize=12)
        ax.set_yticks([])
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add quantile information
        quantiles = metrics_df[metric].quantile([0.25, 0.5, 0.75])
        for q_val, q_name in zip(quantiles, ['Q1', 'Median', 'Q3']):
            ax.axvline(x=q_val, color='blue', linestyle='--', alpha=0.7, linewidth=1)
            ax.text(q_val, -0.3, q_name, ha='center', va='top', 
                   fontsize=8, color='blue', rotation=90)
    
    plt.tight_layout()
    plt.show()

def create_compact_image_line(metrics_df, masks, n_samples=8, figsize=(16, 12)):
    """
    Compact version showing all metrics in one figure with smaller images
    """
    n_metrics = len(metrics_df.columns)
    
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(n_metrics, 1, figure=fig)
    
    for idx, metric in enumerate(metrics_df.columns):
        ax = fig.add_subplot(gs[idx])
        
        # Get metric values and indices
        metric_values = metrics_df[metric].values
        indices = metrics_df.index.values
        
        # Create evenly spaced samples across the range
        x_min, x_max = metric_values.min(), metric_values.max()
        x_linspace = np.linspace(x_min, x_max, n_samples)
        
        # Find closest actual data points
        sample_indices = []
        sample_values = []
        
        for x_val in x_linspace:
            closest_idx = np.argmin(np.abs(metric_values - x_val))
            sample_indices.append(indices[closest_idx])
            sample_values.append(metric_values[closest_idx])
        
        # Setup plot
        ax.set_xlim(x_min - 0.05, x_max + 0.05)
        ax.set_ylim(-1, 1)
        ax.axhline(y=0, color='black', linewidth=1.5)
        
        # Plot points and images
        for i, (x_val, sample_idx) in enumerate(zip(sample_values, sample_indices)):
            # Plot point
            ax.plot(x_val, 0, 'o', markersize=6, color='red', zorder=3)
            
            # Add value
            ax.text(x_val, 0.2, f'{x_val:.2f}', 
                   ha='center', va='bottom', fontsize=8)
            
            # Add mask image
            if sample_idx < len(masks):
                img = masks[sample_idx]
                # Normalize image for better visualization
                img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-8)
                imagebox = OffsetImage(img_normalized, cmap='plasma', zoom=0.2)
                ab = AnnotationBbox(imagebox, (x_val, 0.6), 
                                  frameon=False,
                                  boxcoords="data")
                ax.add_artist(ab)
        
        ax.set_title(f'{metric}', fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel('')
        ax.set_yticks([])
        ax.grid(True, alpha=0.2, axis='x')
    
    plt.tight_layout()
    plt.show()

def create_interactive_like_plot(metrics_df, masks, target_metric=None, n_samples=6):
    """
    Create a focused plot for a specific metric with larger images
    """
    if target_metric is None:
        target_metric = metrics_df.columns[0]
    
    metric_values = metrics_df[target_metric].values
    indices = metrics_df.index.values
    
    fig, ax = plt.subplots(1, 1, figsize=(20, 6))
    
    # Create samples
    x_min, x_max = metric_values.min(), metric_values.max()
    x_linspace = np.linspace(x_min, x_max, n_samples)
    
    sample_indices = []
    sample_values = []
    
    for x_val in x_linspace:
        closest_idx = np.argmin(np.abs(metric_values - x_val))
        sample_indices.append(indices[closest_idx])
        sample_values.append(metric_values[closest_idx])
    
    # Setup main axis
    ax.set_xlim(x_min - 0.1, x_max + 0.1)
    ax.set_ylim(-0.5, 2)
    ax.axhline(y=0, color='black', linewidth=3)
    
    # Plot distribution as background
    hist_data = ax.hist(metric_values, bins=20, alpha=0.3, color='gray', 
                       density=True, zorder=1)
    
    # Add points and images
    for i, (x_val, sample_idx) in enumerate(zip(sample_values, sample_indices)):
        # Plot point on number line
        ax.plot(x_val, 0, 'o', markersize=10, color='red', zorder=4)
        
        # Add value label
        ax.text(x_val, -0.2, f'{x_val:.3f}', 
               ha='center', va='top', fontsize=10, fontweight='bold')
        
        # Add mask image
        if sample_idx < len(masks):
            img = masks[sample_idx]
            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-8)
            imagebox = OffsetImage(img_normalized, cmap='viridis', zoom=0.4)
            ab = AnnotationBbox(imagebox, (x_val, 1.2), 
                              frameon=True,
                              boxcoords="data",
                              pad=0.2,
                              bboxprops=dict(facecolor='white', edgecolor='black', boxstyle="round,pad=0.3"))
            ax.add_artist(ab)
            
            # Add index label above image
            ax.text(x_val, 1.8, f'Idx: {sample_idx}', 
                   ha='center', va='bottom', fontsize=9)
    
    ax.set_title(f'Metric: {target_metric}\nDistribution with Sample Masks', 
                fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Metric Value', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# # Usage example:
# if __name__ == "__main__":
#     # Generate sample data
#     np.random.seed(42)
#     n_samples = 50
    
#     # Sample metrics dataframe
#     metrics_df = pd.DataFrame({
#         'metric_1': np.random.normal(0, 1, n_samples),
#         'metric_2': np.random.exponential(1, n_samples),
#         'metric_3': np.random.uniform(-2, 2, n_samples)
#     })
    
#     # Sample masks (224x224 images)
#     masks = np.random.rand(n_samples, 224, 224)
    
print("Creating comprehensive image line plots...")

# Option 1: Individual plots for each metric
create_quantile_image_line(metrics_df, masks, n_samples=8)

# Option 2: Compact version
create_compact_image_line(metrics_df, masks, n_samples=6)

# Option 3: Focus on one metric
create_interactive_like_plot(metrics_df, masks, target_metric='metric_1', n_samples=5)

In [ ]:
masks = np.array(list(dataset.organoid_masks.values())) 
metrics = get_morphological_features(masks)
metrics_df = pd.DataFrame(metrics)
metrics_df  
print("Creating comprehensive image line plots...")

# Option 1: Individual plots for each metric
create_quantile_image_line(metrics_df, masks, n_samples=8)

In [ ]:
metrics_df